<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/Final_Bayesian_Methodology_Companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Severity Companion Model
Methodology-aligned template.

In [ ]:
# Install
!pip -q install pymc arviz sentence-transformers scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import json
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

from google.colab import drive
drive.mount('/content/drive')
import os

project_folder="/content/drive/MyDrive/Work Place Safety Insights"

train_df=pd.read_csv(f"{project_folder}/train.csv")
test_df=pd.read_csv(f"{project_folder}/test.csv")
transformer_predictions=pd.read_csv(f"{project_folder}/transformer_predictions.csv")

with open(f"{project_folder}/class_weights.json") as f:
    class_weights=json.load(f)


Mounted at /content/drive


## MiniLM Embeddings + PCA

In [ ]:
encoder=SentenceTransformer("all-MiniLM-L6-v2")

train_emb=encoder.encode(train_df["description"].tolist(),show_progress_bar=True)
test_emb=encoder.encode(test_df["description"].tolist(),show_progress_bar=True)

pca=PCA(n_components=100,random_state=42)
X_train=pca.fit_transform(train_emb)
X_test=pca.transform(test_emb)

y_train=train_df["severity_label"].values
y_test=test_df["severity_label"].values


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

## Hierarchical indices

In [ ]:
train_df["sector_idx"],sector_names=pd.factorize(train_df["Industry Sector"])
test_df["sector_idx"]=pd.Categorical(test_df["Industry Sector"],categories=sector_names).codes

train_df["plant_idx"],plant_names=pd.factorize(train_df["Local"])
test_df["plant_idx"]=pd.Categorical(test_df["Local"],categories=plant_names).codes


## Hierarchical Ordinal Bayesian Model

In [ ]:
n_features=X_train.shape[1]
n_classes=len(np.unique(y_train))
n_sectors=len(sector_names)
n_plants=len(plant_names)

with pm.Model() as ordinal_model:

    tau=pm.HalfCauchy("tau",1)

    lam=pm.HalfCauchy("lambda",1,shape=n_features)

    beta=pm.Normal("beta",0,tau*lam,shape=n_features)

    sector_sd=pm.HalfNormal("sector_sd",1)
    plant_sd=pm.HalfNormal("plant_sd",1)

    sector_effect=pm.Normal("sector_effect",0,sector_sd,shape=n_sectors)
    plant_effect=pm.Normal("plant_effect",0,plant_sd,shape=n_plants)

    eta=(pm.math.dot(X_train,beta)
          +sector_effect[train_df["sector_idx"].values]
          +plant_effect[train_df["plant_idx"].values])

    # Redefine cutpoints to ensure ordering explicitly and robustly
    # First cutpoint with a flexible prior
    cutpoints_0 = pm.Normal("cutpoints_0", mu=0, sigma=2)

    # Positive differences between subsequent cutpoints
    # We need n_classes - 1 total cutpoints. If cutpoints_0 is the first, we need (n_classes - 1) - 1 = n_classes - 2 differences.
    # For n_classes = 5, we need 5 - 2 = 3 differences.
    cutpoint_diffs = pm.HalfNormal("cutpoint_diffs", sigma=1, shape=n_classes - 2)

    # Construct the ordered cutpoints using cumulative sum
    cutpoints = pm.Deterministic("cutpoints",
                                 pm.math.concatenate([[cutpoints_0], cutpoints_0 + pm.math.cumsum(cutpoint_diffs)]))

    severity=pm.OrderedLogistic(
        "severity",
        eta=eta,
        cutpoints=cutpoints,
        observed=y_train)

    trace=pm.sample(1000,tune=1000,target_accept=0.95,return_inferencedata=True)

    posterior_pred=pm.sample_posterior_predictive(trace)


Output()

ERROR:pymc.stats.convergence:There were 252 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Output()

## Posterior summaries

In [ ]:
summary=az.summary(trace,hdi_prob=0.95)
summary.to_csv(f"{project_folder}/posterior_summary.csv")
display(summary.head())


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],0.000,0.007,-0.014,0.012,0.001,0.002,483.0,50.0,1.65
beta[1],-0.031,0.229,-0.055,0.031,0.024,0.100,465.0,62.0,1.74
beta[2],-0.000,0.007,-0.015,0.013,0.000,0.001,643.0,200.0,1.70
beta[3],-0.001,0.019,-0.019,0.022,0.001,0.005,1165.0,67.0,1.80
beta[4],0.002,0.028,-0.048,0.028,0.004,0.010,139.0,46.0,1.57


## Posterior prediction on test set (replace with full predictive routine as needed)

In [ ]:
# Expected utility / alert threshold example
# Compute posterior predictive probabilities for test observations
# (extend using pm.Data for production inference)

from google.colab import drive
drive.mount('/content/drive')
import os

project_folder = "/content/drive/MyDrive/Work Place Safety Insights"
os.makedirs(project_folder, exist_ok=True)

decision_results=transformer_predictions.copy()
decision_results["alert"]=decision_results["confidence"]>0.70

decision_results.to_csv(
    f"{project_folder}/decision_layer_results.csv",
    index=False)

transformer_predictions.to_csv(
    f"{project_folder}/bayesian_predictions.csv",
    index=False)

print("Outputs saved.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs saved.
